<a href="https://colab.research.google.com/github/Arpit11-svg/DL/blob/main/TextClassification_%26_seq2seq_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LSTM for Text Classification

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.20.0


In [2]:
# only top 10,000 common words
vocab_size = 10000

# Load IMDB dataset
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Testing samples: 25000


In [3]:
max_length = 200

x_train = pad_sequences(x_train, maxlen=max_length)
x_test = pad_sequences(x_test, maxlen=max_length)

print(x_train.shape)
print(x_test.shape)

(25000, 200)
(25000, 200)


In [4]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_length),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(
    x_train,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 53s 163ms/step - accuracy: 0.7566 - loss: 0.4892 - val_accuracy: 0.8494 - val_loss: 0.3760
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 52s 168ms/step - accuracy: 0.8858 - loss: 0.2852 - val_accuracy: 0.8676 - val_loss: 0.3153
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 86s 182ms/step - accuracy: 0.9269 - loss: 0.1928 - val_accuracy: 0.8608 - val_loss: 0.3538


In [6]:
loss, accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - accuracy: 0.8602 - loss: 0.3565
Test Accuracy: 0.8602399826049805


In [7]:
prediction = model.predict(x_test[:5])

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
[[0.22694829]
 [0.9976096 ]
 [0.94136614]
 [0.35371962]
 [0.9998051 ]]


Values near:

1 → Positive

0 → Negative

LSTM for Sequence-to-Sequence

In [8]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
input_texts = [
    "hello",
    "how are you",
    "good morning",
    "thank you"
]

target_texts = [
    "bonjour",
    "comment ca va",
    "bonjour",
    "merci"
]

In [10]:
input_tokenizer = Tokenizer()
target_tokenizer = Tokenizer()

input_tokenizer.fit_on_texts(input_texts)
target_tokenizer.fit_on_texts(target_texts)

input_sequences = input_tokenizer.texts_to_sequences(input_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)

In [11]:
max_input_len = max(len(seq) for seq in input_sequences)
max_target_len = max(len(seq) for seq in target_sequences)

encoder_input_data = pad_sequences(input_sequences, maxlen=max_input_len, padding='post')
decoder_input_data = pad_sequences(target_sequences, maxlen=max_target_len, padding='post')

In [12]:
input_vocab_size = len(input_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1

latent_dim = 64

In [13]:
encoder_inputs = Input(shape=(None,))
encoder_embedding = tf.keras.layers.Embedding(input_vocab_size, latent_dim)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)

encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

In [14]:
decoder_inputs = Input(shape=(None,))
decoder_embedding = tf.keras.layers.Embedding(target_vocab_size, latent_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(target_vocab_size, activation='softmax')

decoder_outputs = decoder_dense(decoder_outputs)

In [15]:
seq2seq_model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

seq2seq_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

seq2seq_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 64)  │        512 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 64)  │        384 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 64),      │     33,024 │ embedding_1[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, None,     │     33,024 │ embedding_2[0][0… │
│                     │ 64), (None, 64),  │            │ lstm_1[0][1],     │
│                     │ (None, 64)]       │            │ lstm_1[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 6)   │        390 │ lstm_2[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 67,334 (263.02 KB)

 Trainable params: 67,334 (263.02 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
decoder_target_data = np.expand_dims(decoder_input_data, -1)

In [17]:
seq2seq_model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    epochs=100
)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0833 - loss: 1.7972
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5833 - loss: 1.7842
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5000 - loss: 1.7713
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5000 - loss: 1.7585
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5000 - loss: 1.7455
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5000 - loss: 1.7320
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5000 - loss: 1.7178
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.5000 - loss: 1.7028
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.5000 - loss: 1.6867
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5000 - loss: 1.6694
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.5000 - loss: 1.6506
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5000 - loss

In [18]:
predictions = seq2seq_model.predict(
    [encoder_input_data, decoder_input_data]
)

print(predictions.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
(4, 3, 6)


In [19]:
# Reverse dictionary for target words
reverse_target_index = {
    v: k for k, v in target_tokenizer.word_index.items()
}

# Get predicted word indices
predicted_sequences = np.argmax(predictions, axis=-1)

# Print predictions
for i in range(len(predicted_sequences)):
    print("\nInput:", input_texts[i])

    predicted_sentence = []

    for idx in predicted_sequences[i]:
        if idx != 0:
            predicted_sentence.append(
                reverse_target_index.get(idx, "")
            )

    print("Predicted Output:", " ".join(predicted_sentence))


Input: hello
Predicted Output: bonjour

Input: how are you
Predicted Output: comment ca va

Input: good morning
Predicted Output: bonjour

Input: thank you
Predicted Output: merci
